Packages

In [13]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

Paths to Result Summaries

In [14]:
summary_fed_path = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/Extension LLM Classification Task/Fed Side/Results")
summary_ecb_path = Path("/Users/lorenzouberti/Desktop/BANQUE DE FRANCE /Github Repos/Project/BdF-Project/Extension LLM Classification Task/ECB Side/results")
# Summary FED. 
summary_fed_statements_pressconf = pd.read_csv(summary_fed_path / "summary__fed_statements_presconf.csv")
summary_fed_speeches = pd.read_csv(summary_fed_path / "summary__fed_speeches.csv" )
# Summary ECB
summary_ecb_statements_pressconf = pd.read_csv(summary_ecb_path / "summary__ECB_statements_presconf.csv")
summary_ecb_speeches = pd.read_csv(summary_ecb_path / "summary__ECB_speeches.csv")


In [15]:
summary_ecb_statements_pressconf.head(20)

,model,prompt,n,parsed,errors,accuracy,macro_f1,in_tok,out_tok,cost_usd
0,claude-haiku,p1_analyst_direct,2044,2043,0,0.720,0.701,525434,10220,0.577
1,claude-haiku,p2_analyst_rubric,2044,2043,0,0.696,0.683,917882,10220,0.969
2,claude-haiku,p3_analyst_rules,2044,2043,0,0.664,0.622,576534,143327,1.293
3,claude-haiku,p4_analyst_json,2044,2043,0,0.699,0.663,410970,23676,0.529
4,claude-haiku,p5_analyst_fewshot,2044,2043,0,0.690,0.677,619458,10220,0.671
5,claude-sonnet,p1_analyst_direct,2044,2043,0,0.701,0.685,527478,10369,1.738
6,claude-sonnet,p2_analyst_rubric,2044,2043,0,0.691,0.667,919926,10220,2.913
7,claude-sonnet,p3_analyst_rules,2044,2043,0,0.680,0.655,578578,36703,2.286
8,claude-sonnet,p4_analyst_json,2044,2043,0,0.720,0.707,413014,18396,1.515
9,claude-sonnet,p5_analyst_fewshot,2044,2043,0,0.677,0.660,621502,10220,2.018


In [16]:
print(summary_ecb_statements_pressconf.columns)

Index(['model', 'prompt', 'n', 'parsed', 'errors', 'accuracy', 'macro_f1',
       'in_tok', 'out_tok', 'cost_usd'],
      dtype='str')


In [17]:
summary_ecb_speeches.head(2)

,model,prompt,n,parsed,errors,accuracy,macro_f1,in_tok,out_tok,cost_usd
0,claude-haiku,p1_analyst_direct,1000,1000,0,0.683,0.684,256644,5000,0.282
1,claude-haiku,p2_analyst_rubric,1000,1000,0,0.649,0.652,448644,5000,0.474


In [18]:
# ============================================================
# FED vs ECB — model × prompt comparison on macro-F1
# Paste below your loading cell (uses your existing variables).
# ============================================================
import numpy as np
import pandas as pd

try:
    display  # available in notebooks
except NameError:
    display = print

pd.set_option("display.float_format", lambda v: f"{v:.3f}")

runs = pd.concat(
    [
        summary_fed_statements_pressconf.assign(bank="FED", doc_type="statements_pressconf"),
        summary_fed_speeches.assign(bank="FED", doc_type="speeches"),
        summary_ecb_statements_pressconf.assign(bank="ECB", doc_type="statements_pressconf"),
        summary_ecb_speeches.assign(bank="ECB", doc_type="speeches"),
    ],
    ignore_index=True,
)
runs["dataset"] = runs["bank"] + "_" + runs["doc_type"]

# ------------------------------------------------------------
# A · Raw picture — mean macro-F1 per model, per bank
#     (averaged over the 5 prompts and both doc types)
#     NB: ECB_minus_FED here mixes model skill with dataset
#     difficulty — table C fixes that.
# ------------------------------------------------------------
model_by_bank = (
    runs.pivot_table(index="model", columns="bank", values="macro_f1", aggfunc="mean")
        .assign(ECB_minus_FED=lambda d: d["ECB"] - d["FED"])
        .sort_values("ECB_minus_FED", ascending=False)
)
print("A · mean macro-F1 per model × bank")
display(model_by_bank)

# finer grain: one column per dataset
model_by_dataset = runs.pivot_table(
    index="model", columns="dataset", values="macro_f1", aggfunc="mean"
)
print("A2 · mean macro-F1 per model × dataset")
display(model_by_dataset)

# ------------------------------------------------------------
# B · Prompt lift — your idea:
#     each run minus that model's OWN average on the same
#     dataset. Positive = this prompt pushes the model above
#     its usual level; comparable across models because each
#     model is measured against itself.
# ------------------------------------------------------------
runs["prompt_lift"] = runs["macro_f1"] - runs.groupby(["dataset", "model"])[
    "macro_f1"
].transform("mean")

prompt_lift_by_bank = runs.pivot_table(
    index="prompt", columns="bank", values="prompt_lift", aggfunc="mean"
)
print("B · average prompt lift, net of model strength")
display(prompt_lift_by_bank)

# per-model detail (heatmap-ready): which prompt works for WHICH model
prompt_lift_detail = runs.pivot_table(
    index="model", columns=["bank", "prompt"], values="prompt_lift", aggfunc="mean"
)
print("B2 · prompt lift per model (columns: bank × prompt)")
display(prompt_lift_detail)

# ------------------------------------------------------------
# C · Relative model skill — the same centring trick, flipped:
#     each run minus the DATASET average over all models and
#     prompts. This removes dataset difficulty, so
#     ECB_minus_FED > 0 really means "relatively stronger on
#     ECB than the field", not "ECB was easier".
# ------------------------------------------------------------
runs["rel_skill"] = runs["macro_f1"] - runs.groupby("dataset")["macro_f1"].transform("mean")

rel_skill_by_bank = (
    runs.pivot_table(index="model", columns="bank", values="rel_skill", aggfunc="mean")
        .assign(ECB_minus_FED=lambda d: d["ECB"] - d["FED"])
        .sort_values("ECB_minus_FED", ascending=False)
)
print("C · relative skill (dataset-demeaned macro-F1)")
display(rel_skill_by_bank)

# ------------------------------------------------------------
# Extra · best prompt per model & bank (on average macro-F1)
# ------------------------------------------------------------
mp = runs.groupby(["bank", "model", "prompt"], as_index=False)["macro_f1"].mean()
best_prompt = (
    mp.loc[mp.groupby(["bank", "model"])["macro_f1"].idxmax()]
      .sort_values(["bank", "macro_f1"], ascending=[True, False])
      .reset_index(drop=True)
)
print("Best prompt per model × bank")
display(best_prompt)

# ============================================================
# FED vs ECB — D · median robustness, E · document weighting,
#              F · sanity checks implied by the construction
# Paste below the comparison cell. Requires from that cell:
#   runs   (with columns prompt_lift and rel_skill already added)
#   model_by_bank, prompt_lift_by_bank, rel_skill_by_bank
# ============================================================
import numpy as np
import pandas as pd

try:
    display  # available in notebooks
except NameError:
    display = print

# ------------------------------------------------------------
# D · Robustness — medians instead of means
#     One catastrophic run (e.g. JSON parsing failure) drags a
#     10-run mean by drop/10 but barely moves the median
#     (breakdown point ≈ 50% vs 0% for the mean). If the
#     rankings below match tables A and C, the mean results are
#     not outlier-driven.
# ------------------------------------------------------------

# D1 · table A re-run with aggfunc="median"
model_by_bank_median = (
    runs.pivot_table(index="model", columns="bank", values="macro_f1", aggfunc="median")
        .assign(ECB_minus_FED=lambda d: d["ECB"] - d["FED"])
        .sort_values("ECB_minus_FED", ascending=False)
)
print("D1 · median macro-F1 per model × bank")
display(model_by_bank_median)

# D2 · table C re-run with aggfunc="median"
#      (mean-demeaned rel_skill, median-aggregated — the literal
#       re-run suggested in the notes)
rel_skill_by_bank_median = (
    runs.pivot_table(index="model", columns="bank", values="rel_skill", aggfunc="median")
        .assign(ECB_minus_FED=lambda d: d["ECB"] - d["FED"])
        .sort_values("ECB_minus_FED", ascending=False)
)
print("D2 · median relative skill per model × bank")
display(rel_skill_by_bank_median)

# D2b · fully robust alternative: demean by the DATASET MEDIAN,
#       then aggregate by median, so no mean enters anywhere.
runs["rel_skill_med"] = runs["macro_f1"] - runs.groupby("dataset")["macro_f1"].transform("median")
rel_skill_med_by_bank = (
    runs.pivot_table(index="model", columns="bank", values="rel_skill_med", aggfunc="median")
        .assign(ECB_minus_FED=lambda d: d["ECB"] - d["FED"])
        .sort_values("ECB_minus_FED", ascending=False)
)
print("D2b · fully median-based relative skill (median-demeaned, median-aggregated)")
display(rel_skill_med_by_bank)

# D3 · is the model ranking stable under mean vs median?
#      Spearman rho computed as Pearson on ranks (no scipy needed).
stability = pd.DataFrame({
    "gap_mean": model_by_bank["ECB_minus_FED"],
    "gap_median": model_by_bank_median["ECB_minus_FED"],
})
stability["rank_mean"] = stability["gap_mean"].rank(ascending=False, method="min").astype(int)
stability["rank_median"] = stability["gap_median"].rank(ascending=False, method="min").astype(int)
spearman_rho = stability["gap_mean"].rank().corr(stability["gap_median"].rank())
print("D3 · rank stability of ECB_minus_FED (1 = most ECB-leaning)")
display(stability.sort_values("rank_mean"))
print(f"Spearman rho (mean vs median ranking) = {spearman_rho:.3f}")

# D4 · screen for tanked runs — inspect these raw outputs before
#      trusting any mean-based table.
TANK_THRESHOLD = -0.05  # macro-F1 points below the model's own (model, dataset) mean; adjust
tanked = (
    runs.loc[runs["prompt_lift"] < TANK_THRESHOLD,
             ["model", "dataset", "prompt", "macro_f1", "prompt_lift"]]
        .sort_values("prompt_lift")
)
print(f"D4 · runs more than {-TANK_THRESHOLD:.2f} below their own (model, dataset) mean: {len(tanked)}")
display(tanked)

# ------------------------------------------------------------
# E · Document-weighted aggregation
#     Equal-weight means treat each corpus as one task; document
#     weighting treats each document equally, so statements
#     (~2044 docs) count ≈2× speeches (~1000). n varies only at
#     the dataset level, so the demeaning behind rel_skill is
#     UNCHANGED — only the cross-doc-type aggregation reweights.
#     NB: a document-weighted average of per-corpus macro-F1 is
#     NOT the macro-F1 of the pooled corpus (F1 is a ratio metric
#     and does not decompose across corpora). Report it as a
#     weighted average of per-corpus scores.
# ------------------------------------------------------------

if "n" not in runs.columns:
    # TODO: replace with your true document counts per (bank, doc_type)
    n_docs = {
        ("FED", "statements_pressconf"): 2044,
        ("FED", "speeches"): 1000,
        ("ECB", "statements_pressconf"): 2044,
        ("ECB", "speeches"): 1000,
    }
    runs["n"] = [n_docs[(b, t)] for b, t in zip(runs["bank"], runs["doc_type"])]


def weighted_by_bank(df, value_col):
    """n-weighted mean of value_col per model × bank.

    Sum-of-products form: avoids groupby.apply, so it is stable
    across pandas versions and raises no deprecation warnings.
    """
    tmp = df.assign(_wx=df[value_col] * df["n"])
    g = tmp.groupby(["model", "bank"])[["_wx", "n"]].sum()
    return (g["_wx"] / g["n"]).unstack("bank")


# E1 · document-weighted table A
model_by_bank_weighted = (
    weighted_by_bank(runs, "macro_f1")
    .assign(ECB_minus_FED=lambda d: d["ECB"] - d["FED"])
    .sort_values("ECB_minus_FED", ascending=False)
)
print("E1 · document-weighted mean macro-F1 per model × bank")
display(model_by_bank_weighted)

# E2 · document-weighted table C
rel_skill_by_bank_weighted = (
    weighted_by_bank(runs, "rel_skill")
    .assign(ECB_minus_FED=lambda d: d["ECB"] - d["FED"])
    .sort_values("ECB_minus_FED", ascending=False)
)
print("E2 · document-weighted relative skill")
display(rel_skill_by_bank_weighted)

# E3 · document-weighted best prompt per model × bank
mp_w = (
    runs.assign(_wx=runs["macro_f1"] * runs["n"])
        .groupby(["bank", "model", "prompt"], as_index=False)[["_wx", "n"]].sum()
        .assign(macro_f1_w=lambda d: d["_wx"] / d["n"])
)
best_prompt_weighted = (
    mp_w.loc[mp_w.groupby(["bank", "model"])["macro_f1_w"].idxmax(),
             ["bank", "model", "prompt", "macro_f1_w"]]
        .sort_values(["bank", "macro_f1_w"], ascending=[True, False])
        .reset_index(drop=True)
)
print("E3 · document-weighted best prompt per model × bank")
display(best_prompt_weighted)

# ------------------------------------------------------------
# F · Numerical sanity checks implied by the construction
#     Within-cell sums are identically zero however many runs
#     exist; the bank-level column sums are exactly zero only on
#     a complete model × prompt × dataset grid.
# ------------------------------------------------------------
assert np.allclose(runs.groupby(["dataset", "model"])["prompt_lift"].sum(), 0, atol=1e-9), \
    "prompt_lift must sum to 0 within every (dataset, model) cell"
assert np.allclose(runs.groupby("dataset")["rel_skill"].sum(), 0, atol=1e-9), \
    "rel_skill must sum to 0 within every dataset"

print("F · column sums (≈ 0 by construction on a complete grid):")
print("    B, lift per bank:      ", prompt_lift_by_bank.sum().round(6).to_dict())
print("    C, rel. skill per bank:", rel_skill_by_bank[["FED", "ECB"]].sum().round(6).to_dict())

A · mean macro-F1 per model × bank


bank,ECB,FED,ECB_minus_FED
model,,,
deepseek-reasoner,0.706,0.653,0.053
mistral-large,0.668,0.624,0.044
deepseek-chat,0.593,0.550,0.044
claude-sonnet,0.677,0.635,0.042
claude-haiku,0.660,0.629,0.031
gpt-5.1,0.665,0.646,0.019
mistral-small,0.610,0.595,0.015
gpt-5-mini,0.634,0.635,-0.001


A2 · mean macro-F1 per model × dataset


dataset,ECB_speeches,ECB_statements_pressconf,FED_speeches,FED_statements_pressconf
model,,,,
claude-haiku,0.651,0.669,0.623,0.636
claude-sonnet,0.678,0.675,0.660,0.610
deepseek-chat,0.563,0.623,0.561,0.538
deepseek-reasoner,0.713,0.699,0.665,0.641
gpt-5-mini,0.604,0.663,0.630,0.640
gpt-5.1,0.636,0.695,0.648,0.644
mistral-large,0.655,0.681,0.627,0.620
mistral-small,0.600,0.619,0.593,0.596


B · average prompt lift, net of model strength


...

E2 · document-weighted relative skill


bank,ECB,FED,ECB_minus_FED
model,,,
deepseek-chat,-0.053,-0.073,0.021
deepseek-reasoner,0.047,0.030,0.017
claude-sonnet,0.020,0.004,0.015
mistral-large,0.016,0.004,0.013
claude-haiku,0.007,0.013,-0.006
gpt-5.1,0.019,0.026,-0.007
mistral-small,-0.044,-0.024,-0.020
gpt-5-mini,-0.013,0.018,-0.030


E3 · document-weighted best prompt per model × bank


,bank,model,prompt,macro_f1_w
0,ECB,deepseek-reasoner,p4_analyst_json,0.726
1,ECB,gpt-5.1,p4_analyst_json,0.709
2,ECB,claude-sonnet,p4_analyst_json,0.703
3,ECB,mistral-large,p4_analyst_json,0.697
4,ECB,claude-haiku,p1_analyst_direct,0.695
5,ECB,gpt-5-mini,p1_analyst_direct,0.678
6,ECB,deepseek-chat,p4_analyst_json,0.673
7,ECB,mistral-small,p4_analyst_json,0.665
8,FED,deepseek-reasoner,p2_analyst_rubric,0.669
9,FED,gpt-5-mini,p1_analyst_direct,0.665


F · column sums (≈ 0 by construction on a complete grid):
    B, lift per bank:       {'ECB': -0.0, 'FED': -0.0}
    C, rel. skill per bank: {'FED': -0.0, 'ECB': 0.0}
